# PatchTST: Goldkursanalyse

Dieses Notebook widmet sich dem Suchen der besten Hyperparameter für PatchTST


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "torch": "torch",
}

missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already available.")

All required packages are already available.


In [2]:
import math
import random
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
import pandas as pd

df = pd.read_csv("../Data/goldpreis_seit_2000_prep.csv")

df["Date"] = pd.to_datetime(df["Date"],utc=True)
df = df.sort_values("Date").reset_index(drop=True)

print(df.shape)
display(df.head())
display(df.tail())

(6491, 10)


,Unnamed: 0,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Change %
0,0,2000-09-01 04:00:00+00:00,277.000000,277.000000,277.000000,277.000000,0,0.0,0.0,0.000000
1,1,2000-09-05 04:00:00+00:00,275.799988,275.799988,275.799988,275.799988,2,0.0,0.0,-0.004332
2,2,2000-09-06 04:00:00+00:00,274.200012,274.200012,274.200012,274.200012,0,0.0,0.0,-0.005801
3,3,2000-09-07 04:00:00+00:00,274.000000,274.000000,274.000000,274.000000,125,0.0,0.0,-0.000729
4,4,2000-09-08 04:00:00+00:00,273.299988,273.299988,273.299988,273.299988,0,0.0,0.0,-0.002555


,Unnamed: 0,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Change %
6486,6486,2026-07-13 04:00:00+00:00,4081.000000,4081.000000,3985.899902,3997.000000,679,0.0,0.0,-0.026096
6487,6487,2026-07-14 04:00:00+00:00,3995.699951,4091.199951,3986.500000,4061.100098,1281,0.0,0.0,0.016037
6488,6488,2026-07-15 04:00:00+00:00,4049.100098,4070.100098,4019.399902,4044.000000,374,0.0,0.0,-0.004211
6489,6489,2026-07-16 04:00:00+00:00,4030.500000,4030.500000,3972.600098,3985.600098,374,0.0,0.0,-0.014441
6490,6490,2026-07-17 04:00:00+00:00,3980.100098,4012.199951,3963.000000,3999.199951,70785,0.0,0.0,0.003412


## 2. Data Understanding

Wir verwenden den Goldkurs seit September 2000

Warum eignet sich dieser Datensatz für PatchTST?

- Er ist eine echte, regelmäßig gemessene Zeitreihe.
- Er enthält einen numerische Kanal, wodurch PatchTST ggf. leicht überdimensioniert ist.
- Er besitzt genügend Länge für Sliding Windows und Patches.
- Die Zielvariable `Close` wird für unseren Anwendungsfall genutzt.

In [29]:
import pandas as pd

df = pd.read_csv("../Data/goldpreis_seit_2000_prep.csv")

df["Date"] = pd.to_datetime(df["Date"],utc=True)
df = df.sort_values("Date").reset_index(drop=True)

print(df.shape)
display(df.head())
display(df.tail())

(6491, 10)


,Unnamed: 0,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Change %
0,0,2000-09-01 04:00:00+00:00,277.000000,277.000000,277.000000,277.000000,0,0.0,0.0,0.000000
1,1,2000-09-05 04:00:00+00:00,275.799988,275.799988,275.799988,275.799988,2,0.0,0.0,-0.004332
2,2,2000-09-06 04:00:00+00:00,274.200012,274.200012,274.200012,274.200012,0,0.0,0.0,-0.005801
3,3,2000-09-07 04:00:00+00:00,274.000000,274.000000,274.000000,274.000000,125,0.0,0.0,-0.000729
4,4,2000-09-08 04:00:00+00:00,273.299988,273.299988,273.299988,273.299988,0,0.0,0.0,-0.002555


,Unnamed: 0,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Change %
6486,6486,2026-07-13 04:00:00+00:00,4081.000000,4081.000000,3985.899902,3997.000000,679,0.0,0.0,-0.026096
6487,6487,2026-07-14 04:00:00+00:00,3995.699951,4091.199951,3986.500000,4061.100098,1281,0.0,0.0,0.016037
6488,6488,2026-07-15 04:00:00+00:00,4049.100098,4070.100098,4019.399902,4044.000000,374,0.0,0.0,-0.004211
6489,6489,2026-07-16 04:00:00+00:00,4030.500000,4030.500000,3972.600098,3985.600098,374,0.0,0.0,-0.014441
6490,6490,2026-07-17 04:00:00+00:00,3980.100098,4012.199951,3963.000000,3999.199951,70785,0.0,0.0,0.003412


### Spalten und Zeitindex

Jede Zeile repräsentiert eine Stunde. Die Spalte `Date` ist der Zeitstempel. Alle übrigen Spalten sind numerische Messgrößen. Wir verwenden später `Close` als Zielvariable. Die anderen Kanäle werden nicht beachtet.

In [30]:
print(df.info())
print("\nColumns:", list(df.columns))
print("Time range:", df["Date"].min(), "to", df["Date"].max())
print("Median time difference:", df["Date"].diff().median())

<class 'pandas.DataFrame'>
RangeIndex: 6491 entries, 0 to 6490
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   Unnamed: 0    6491 non-null   int64              
 1   Date          6491 non-null   datetime64[us, UTC]
 2   Open          6491 non-null   float64            
 3   High          6491 non-null   float64            
 4   Low           6491 non-null   float64            
 5   Close         6491 non-null   float64            
 6   Volume        6491 non-null   int64              
 7   Dividends     6491 non-null   float64            
 8   Stock Splits  6491 non-null   float64            
 9   Change %      6491 non-null   float64            
dtypes: datetime64[us, UTC](1), float64(7), int64(2)
memory usage: 507.2 KB
None

Columns: ['Unnamed: 0', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Change %']
Time range: 2000-09-01 04:00:00+00:00 to 2026-0

## 3. Data Preparation

Wir bereiten die Daten so vor, dass keine Information aus der Zukunft ins Training gelangt:

- Split nach Zeitreihenfolge, nicht zufällig.
- Normalisierung nur mit Statistiken des Trainingsbereichs.
- Sliding Windows für Eingabe und Prognoseziel.
- Tensoren im Format `(samples, lookback, channels)` und `(samples, horizon, channels)`.

Wichtige Begriffe:

- **Lookback window:** Wie viele vergangene Zeitschritte das Modell sieht.
- **Forecast horizon:** Wie viele zukünftige Zeitschritte das Modell vorhersagt.

In [6]:
target_column = "Close"
feature_columns = target_column
target_index = feature_columns.index(target_column)

lookback = 96
horizon = 24
patch_length = 16
stride = 8

values = df["Close"].astype("float32").values
dates = df["Date"].values

train_end = int(len(values) * 0.70)
validation_end = int(len(values) * 0.90)

train_values_raw = values[:train_end]
validation_values_raw = values[train_end:validation_end]
test_values_raw = values[validation_end:]

train_dates = dates[:train_end]
validation_dates = dates[train_end:validation_end]
test_dates = dates[validation_end:]

train_mean = train_values_raw.mean(axis=0, keepdims=True)
train_std = train_values_raw.std(axis=0, keepdims=True)
train_std = np.where(train_std == 0, 1.0, train_std)

train_values = (train_values_raw - train_mean) / train_std
validation_values = (validation_values_raw - train_mean) / train_std
test_values = (test_values_raw - train_mean) / train_std

print("Train rows:", train_values.shape)
print("Validation rows:", validation_values.shape)
print("Test rows:", test_values.shape)
print("Number of channels:", len(feature_columns))

Train rows: (4543,)
Validation rows: (1298,)
Test rows: (650,)
Number of channels: 5


### Sliding Windows

Ein Sliding Window erzeugt viele Trainingsbeispiele aus einer langen Zeitreihe. Für jedes Beispiel nimmt man einen Vergangenheitsblock der Länge `lookback` und den direkt folgenden Zukunftsblock der Länge `horizon`.

In [7]:
def create_sliding_windows(data, lookback_length, horizon_length):
    # Create input and target windows from a multivariate time series.
    inputs = []
    targets = []
    max_start = len(data) - lookback_length - horizon_length + 1

    for start in range(max_start):
        end = start + lookback_length
        target_end = end + horizon_length
        inputs.append(data[start:end])
        targets.append(data[end:target_end])


    return np.stack(inputs).astype("float32"), np.stack(targets).astype("float32")


X_train, y_train = create_sliding_windows(train_values, lookback, horizon)
X_validation, y_validation = create_sliding_windows(validation_values, lookback, horizon)
X_test, y_test = create_sliding_windows(test_values, lookback, horizon)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (4424, 96)
y_train: (4424, 24)
X_validation: (1179, 96)
y_validation: (1179, 24)
X_test: (531, 96)
y_test: (531, 24)


In [8]:
X_train = X_train[:, :, np.newaxis]
y_train = y_train[:, :, np.newaxis]
X_validation =  X_validation[:, :, np.newaxis]
y_validation = y_validation[:, :, np.newaxis]
X_test = X_test[:, :, np.newaxis]
y_test = y_test[:, :, np.newaxis]

print("X_train:",X_train.shape)      
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


X_train: (4424, 96, 1)
y_train: (4424, 24, 1)
X_validation: (1179, 96, 1)
y_validation: (1179, 24, 1)
X_test: (531, 96, 1)
y_test: (531, 24, 1)


## 5. Modeling

Wir implementieren eine kleine, lesbare PatchTST-Variante in PyTorch. Sie ist bewusst kompakt:

- Patching mit `unfold`
- lineares Patch Embedding
- lernbare Positionskodierung
- Transformer Encoder
- lineare Vorhersage für den Forecast Horizon

Die Ausgabe hat die Form `(batch, horizon, channels)`.

In [10]:
from dataclasses import dataclass
import torch
import torch.nn as nn

@dataclass
class PatchTSTConfig:
    lookback: int
    horizon: int
    num_channels: int
    patch_length: int
    stride: int
    d_model: int = 64
    n_heads: int = 4
    num_layers: int = 2
    dropout: float = 0.10

    @property
    def num_patches(self):
        return ((self.lookback - self.patch_length) // self.stride) + 1


class EducationalPatchTST(nn.Module):
    def __init__(self, lookback, horizon, num_channels, patch_length, stride, d_model, n_heads):
        super().__init__()
        
        # Zuweisung Variablen
        self.lookback = lookback
        self.horizon = horizon
        self.num_channels = num_channels
        self.patch_length = patch_length
        self.stride = stride
        self.d_model = d_model
        self.n_heads = n_heads
        self.num_layers = 4  
        dropout = 0.10

        # Layer & Parameter Definitionen
        self.patch_embedding = nn.Linear(patch_length, d_model)
        
        # Nutzen der Klassen-Property für das Positions-Embedding
        self.position_embedding = nn.Parameter(torch.zeros(1, self.num_patches, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers)
        self.dropout = nn.Dropout(dropout)
        
        self.prediction_head = nn.Linear(self.num_patches * d_model, horizon)

    @property
    def num_patches(self):
        return ((self.lookback - self.patch_length) // self.stride) + 1

    def forward(self, x, print_shapes=False):
        batch_size, lookback_length, num_channels = x.shape

        if print_shapes:
            print(f"Input: {x.shape} -> (batch, lookback, channels) = ({batch_size}, {lookback_length}, {num_channels})")

        x = x.permute(0, 2, 1)
        x = x.reshape(batch_size * num_channels, lookback_length)

        # Unfolding in Patches
        patches = x.unfold(dimension=-1, size=self.patch_length, step=self.stride)

        if print_shapes:
            print(f"Patches: {patches.shape} -> (batch * channels, num_patches, patch_length) = ({batch_size * num_channels}, {self.num_patches}, {self.patch_length})")

        embedded = self.patch_embedding(patches)
        embedded = embedded + self.position_embedding
        embedded = self.dropout(embedded)

        if print_shapes:
            print(f"Embedded patches: {embedded.shape} -> each raw patch is projected from length {self.patch_length} to d_model={self.d_model}")

        encoded = self.encoder(embedded)

        if print_shapes:
            print(f"Encoded patches: {encoded.shape} -> Transformer keeps the same shape but enriches each patch with attention context")

        # Flatten & Lineare Vorhersage
        flattened = encoded.reshape(batch_size * num_channels, -1)
        forecast = self.prediction_head(flattened)
        
        # Zurückformen in die ursprüngliche Kanalstruktur
        forecast = forecast.reshape(batch_size, num_channels, self.horizon)
        forecast = forecast.permute(0, 2, 1)

        if print_shapes:
            print(f"Forecast: {forecast.shape} -> (batch, horizon, channels) = ({batch_size}, {self.horizon}, {num_channels})")

        return forecast


# --- Instanziierung & Testlauf ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# KORREKTUR: patch_length (12) darf nicht größer sein als lookback (40). d_model korrigiert (z.B. 64).
model = EducationalPatchTST(
    lookback=40, 
    horizon=5, 
    num_channels=1, 
    patch_length=12, 
    stride=4, 
    d_model=64, 
    n_heads=4
).to(device)

# Shapes testen... Dummy
dummy_input = torch.randn(32, 40, 1).to(device)  # Batch=32, Lookback=40, Channels=1
output = model(dummy_input, print_shapes=True)


Input: torch.Size([32, 40, 1]) -> (batch, lookback, channels) = (32, 40, 1)
Patches: torch.Size([32, 8, 12]) -> (batch * channels, num_patches, patch_length) = (32, 8, 12)
Embedded patches: torch.Size([32, 8, 64]) -> each raw patch is projected from length 12 to d_model=64
Encoded patches: torch.Size([32, 8, 64]) -> Transformer keeps the same shape but enriches each patch with attention context
Forecast: torch.Size([32, 5, 1]) -> (batch, horizon, channels) = (32, 5, 1)


### DataLoader

Für eine schnelle Demonstration begrenzen wir die Anzahl der Trainingsfenster. Der Testbereich bleibt unverändert. In einer ernsthaften Modellierung würde man mehr Epochen, Hyperparameter-Suche und eventuell den vollständigen Trainingsbereich verwenden.

In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader, Subset

def get_data_loaders(train_values, validation_values, test_values, batch_size, lookback, horizon):
    """
    Erstellt chronologisch getrennte Datenlader für dynamische Fenstergrößen.
    
    Args:
        batch_size (int): Anzahl der Samples pro Batch.
        lookback_window (int): Länge der historischen Eingangssequenz (z.B. 20, 40, 80, 120).
        prediction_horizon (int): Länge des Vorhersagefensters (z.B. 1, 5, 20).
    """
    X_train, y_train = create_sliding_windows(train_values, lookback, horizon)
    X_validation, y_validation = create_sliding_windows(validation_values, lookback, horizon)
    X_test, y_test = create_sliding_windows(test_values, lookback, horizon)
        
    X_train = X_train[:, :, np.newaxis]
    y_train = y_train[:, :, np.newaxis]

    X_validation =  X_validation[:, :, np.newaxis]
    y_validation = y_validation[:, :, np.newaxis]
    X_test = X_test[:, :, np.newaxis]
    y_test = y_test[:, :, np.newaxis]
    
    train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
    
    )
    validation_dataset = TensorDataset(
    torch.tensor(X_validation, dtype=torch.float32),
    torch.tensor(y_validation, dtype=torch.float32),
    )
    test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
    )
   # print("Done")

    # Loader erstellen
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import optuna

In [13]:
# Hyperparameter Suche für Horizon 1
def objective(trial):
    # --- Hyperparameter-Suchraum definieren ---
    patch_length = trial.suggest_categorical("patch_len", [4, 8, 12, 16, 32])
    stride = trial.suggest_categorical("stride", [2,4, 8, 16])
    d_model = trial.suggest_categorical("d_model", [64, 128, 256])
    n_heads = trial.suggest_categorical("n_heads", [2, 4, 8])
    #seq_length = trial.suggest_categorical("seq_length",[20,30,40,50])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])
    horizon = trial.suggest_categorical("horizon",[1, 5,20] )
    lookback = trial.suggest_categorical("lookback",[30,40,50,60] )
    
    # Daten laden basierend auf der vorgeschlagenen Batch-Größe
    train_loader, val_loader, test_loader = get_data_loaders(train_values, validation_values, test_values, batch_size, lookback, horizon)
    
    # Modell initialisieren
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EducationalPatchTST(
            lookback=lookback, horizon=horizon,num_channels=1,patch_length=patch_length,
            stride=stride,d_model=d_model,n_heads=n_heads
    ).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Training & Validierungsschleife (kurz gehalten für das Tuning)
    for epoch in range(100): # In der Praxis mehr Epochen nutzen
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        # Validierung
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                val_loss += criterion(outputs, batch_y).item()
                
        avg_val_loss = val_loss / len(val_loader)
        
        # Pruning: Schlechte Trials frühzeitig abbrechen
        trial.report(avg_val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    return avg_val_loss

# 4. Optimierung starten
if __name__ == "__main__":
    # Erstellt eine Studie, die den Validierungsverlust minimiert
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=20) # n_trials bestimmt die Anzahl der Testläufe

    print("\n✅ Optimierung abgeschlossen!")
    print("Beste Parameter:", study.best_params)
    print("Niedrigster Validierungsverlust:", study.best_value)


[I 2026-07-18 15:29:14,864] A new study created in memory with name: no-name-43fc3944-9e2d-4432-8c8b-aa1614d69742
[I 2026-07-18 15:32:58,890] Trial 0 finished with value: 0.15678754731570968 and parameters: {'patch_len': 4, 'stride': 8, 'd_model': 64, 'n_heads': 2, 'lr': 0.0003225075525304254, 'batch_size': 32, 'horizon': 5, 'lookback': 50}. Best is trial 0 with value: 0.15678754731570968.
[I 2026-07-18 15:39:10,291] Trial 1 finished with value: 0.10122879651462427 and parameters: {'patch_len': 4, 'stride': 4, 'd_model': 128, 'n_heads': 8, 'lr': 0.00020442522490327045, 'batch_size': 32, 'horizon': 5, 'lookback': 40}. Best is trial 1 with value: 0.10122879651462427.
[I 2026-07-18 15:42:44,493] Trial 2 finished with value: 3.094595006108284 and parameters: {'patch_len': 4, 'stride': 8, 'd_model': 128, 'n_heads': 8, 'lr': 0.003843085200937758, 'batch_size': 64, 'horizon': 20, 'lookback': 40}. Best is trial 1 with value: 0.10122879651462427.
[I 2026-07-18 15:44:13,880] Trial 3 finished wit


✅ Optimierung abgeschlossen!
Beste Parameter: {'patch_len': 8, 'stride': 2, 'd_model': 64, 'n_heads': 4, 'lr': 0.00013475492072702265, 'batch_size': 32, 'horizon': 1, 'lookback': 30}
Niedrigster Validierungsverlust: 0.017900372497388163


Ergebnis der Parametersuche:
Beste Parameter: {'patch_len': 8, 'stride': 2, 'd_model': 64, 'n_heads': 4, 'lr': 0.00013475492072702265, 'batch_size': 32, 'horizon': 1, 'lookback': 30}
Niedrigster Validierungsverlust: 0.017900372497388163

In [20]:
# Hyperparameter Suche für Horizon 5
def objective5(trial):
    # --- Hyperparameter-Suchraum definieren ---
    patch_length = trial.suggest_categorical("patch_len", [4, 8, 12, 16, 32])
    stride = trial.suggest_categorical("stride", [2,4, 8, 16])
    d_model = trial.suggest_categorical("d_model", [64, 128, 256])
    n_heads = trial.suggest_categorical("n_heads", [2, 4, 8])
    #seq_length = trial.suggest_categorical("seq_length",[20,30,40,50])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])
    horizon = trial.suggest_categorical("horizon",[5] )
    lookback = trial.suggest_categorical("lookback",[30,40,50,60] )
    
    # Daten laden basierend auf der vorgeschlagenen Batch-Größe
    train_loader, val_loader, test_loader = get_data_loaders(train_values, validation_values, test_values, batch_size, lookback, horizon)
    
    # Modell initialisieren
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EducationalPatchTST(
            lookback=lookback, horizon=horizon,num_channels=1,patch_length=patch_length,
            stride=stride,d_model=d_model,n_heads=n_heads
    ).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Training & Validierungsschleife (kurz gehalten für das Tuning)
    for epoch in range(100): # In der Praxis mehr Epochen nutzen
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        # Validierung
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                val_loss += criterion(outputs, batch_y).item()
                
        avg_val_loss = val_loss / len(val_loader)
        
        # Pruning: Schlechte Trials frühzeitig abbrechen
        trial.report(avg_val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    return avg_val_loss

# 4. Optimierung starten
if __name__ == "__main__":
    # Erstellt eine Studie, die den Validierungsverlust minimiert
    study5 = optuna.create_study(direction="minimize")
    study5.optimize(objective5, n_trials=20) # n_trials bestimmt die Anzahl der Testläufe

    print("\n✅ Optimierung abgeschlossen!")
    print("Beste Parameter:", study5.best_params)
    print("Niedrigster Validierungsverlust:", study5.best_value)


[I 2026-07-18 16:29:45,027] A new study created in memory with name: no-name-1e5aaae0-0757-4c56-aff0-0e87a69a42e9


[I 2026-07-18 16:43:51,352] Trial 0 finished with value: 0.05026960817631334 and parameters: {'patch_len': 16, 'stride': 2, 'd_model': 256, 'n_heads': 4, 'lr': 0.0009537382946996367, 'batch_size': 64, 'horizon': 5, 'lookback': 60}. Best is trial 0 with value: 0.05026960817631334.
[I 2026-07-18 16:49:40,214] Trial 1 finished with value: 0.041610148688778284 and parameters: {'patch_len': 4, 'stride': 2, 'd_model': 64, 'n_heads': 8, 'lr': 0.00022870374392035072, 'batch_size': 64, 'horizon': 5, 'lookback': 50}. Best is trial 1 with value: 0.041610148688778284.
[I 2026-07-18 16:51:09,558] Trial 2 finished with value: 0.06603766800477527 and parameters: {'patch_len': 8, 'stride': 16, 'd_model': 64, 'n_heads': 8, 'lr': 0.004039432728430289, 'batch_size': 32, 'horizon': 5, 'lookback': 60}. Best is trial 1 with value: 0.041610148688778284.
[I 2026-07-18 16:55:38,202] Trial 3 finished with value: 0.09794379063905814 and parameters: {'patch_len': 4, 'stride': 16, 'd_model': 256, 'n_heads': 2, 'lr


✅ Optimierung abgeschlossen!
Beste Parameter: {'patch_len': 8, 'stride': 2, 'd_model': 128, 'n_heads': 2, 'lr': 0.00010244599589243109, 'batch_size': 64, 'horizon': 5, 'lookback': 50}
Niedrigster Validierungsverlust: 0.015879153972491622


Ergebnis der Parametersuche: 

Beste Parameter: {'patch_len': 8, 'stride': 2, 'd_model': 128, 'n_heads': 2, 'lr': 0.00010244599589243109, 'batch_size': 64, 'horizon': 5, 'lookback': 50}
Niedrigster Validierungsverlust: 0.015879153972491622

In [22]:
# Hyperparameter Suche für Horizon 20
def objective20(trial):
    # --- Hyperparameter-Suchraum definieren ---
    patch_length = trial.suggest_categorical("patch_len", [4, 8, 12, 16, 32])
    stride = trial.suggest_categorical("stride", [2,4, 8, 16])
    d_model = trial.suggest_categorical("d_model", [64, 128, 256])
    n_heads = trial.suggest_categorical("n_heads", [2, 4, 8])
    #seq_length = trial.suggest_categorical("seq_length",[20,30,40,50])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])
    horizon = trial.suggest_categorical("horizon",[20] )
    lookback = trial.suggest_categorical("lookback",[40,50,60] )
    
    # Daten laden basierend auf der vorgeschlagenen Batch-Größe
    train_loader, val_loader, test_loader = get_data_loaders(train_values, validation_values, test_values, batch_size, lookback, horizon)
    
    # Modell initialisieren
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EducationalPatchTST(
            lookback=lookback, horizon=horizon,num_channels=1,patch_length=patch_length,
            stride=stride,d_model=d_model,n_heads=n_heads
    ).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Training & Validierungsschleife (kurz gehalten für das Tuning)
    for epoch in range(100): # In der Praxis mehr Epochen nutzen
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        # Validierung
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                val_loss += criterion(outputs, batch_y).item()
                
        avg_val_loss = val_loss / len(val_loader)
        
        # Pruning: Schlechte Trials frühzeitig abbrechen
        trial.report(avg_val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    return avg_val_loss

# 4. Optimierung starten
if __name__ == "__main__":
    # Erstellt eine Studie, die den Validierungsverlust minimiert
    study20 = optuna.create_study(direction="minimize")
    study20.optimize(objective20, n_trials=20) # n_trials bestimmt die Anzahl der Testläufe

    print("\n✅ Optimierung abgeschlossen!")
    print("Beste Parameter:", study20.best_params)
    print("Niedrigster Validierungsverlust:", study20.best_value)


[I 2026-07-18 17:53:57,744] A new study created in memory with name: no-name-2c84911b-5097-4c3f-8e75-842eaf108c80
[I 2026-07-18 17:59:15,396] Trial 0 finished with value: 3.7051555305719375 and parameters: {'patch_len': 16, 'stride': 4, 'd_model': 128, 'n_heads': 8, 'lr': 0.003179801446480821, 'batch_size': 64, 'horizon': 20, 'lookback': 60}. Best is trial 0 with value: 3.7051555305719375.
[I 2026-07-18 18:02:47,043] Trial 1 finished with value: 0.11499145879643038 and parameters: {'patch_len': 32, 'stride': 4, 'd_model': 128, 'n_heads': 8, 'lr': 0.0003585198260513696, 'batch_size': 64, 'horizon': 20, 'lookback': 50}. Best is trial 1 with value: 0.11499145879643038.
[I 2026-07-18 18:14:26,257] Trial 2 finished with value: 0.05231743356069693 and parameters: {'patch_len': 12, 'stride': 4, 'd_model': 256, 'n_heads': 8, 'lr': 0.0008456993780082472, 'batch_size': 32, 'horizon': 20, 'lookback': 60}. Best is trial 2 with value: 0.05231743356069693.
[I 2026-07-18 18:19:47,006] Trial 3 finishe


✅ Optimierung abgeschlossen!
Beste Parameter: {'patch_len': 12, 'stride': 4, 'd_model': 256, 'n_heads': 8, 'lr': 0.0008456993780082472, 'batch_size': 32, 'horizon': 20, 'lookback': 60}
Niedrigster Validierungsverlust: 0.05231743356069693


Ergebnis der Parametersuche:

Beste Parameter: {'patch_len': 12, 'stride': 4, 'd_model': 256, 'n_heads': 8, 'lr': 0.0008456993780082472, 'batch_size': 32, 'horizon': 20, 'lookback': 60}
Niedrigster Validierungsverlust: 0.05231743356069693